# Module 12 — Greedy Algorithms and Interval Scheduling

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_max_profit_stock import max_profit_stock
from p02_merge_intervals import merge_intervals
from p03_max_non_overlapping import max_non_overlapping

print("module 12: Greedy Algorithms and Interval Scheduling")
print("problems available:", 8)
for name in ['p01_max_profit_stock', 'p02_merge_intervals', 'p03_max_non_overlapping', 'p04_min_arrows', 'p05_can_jump', 'p06_min_jumps', 'p07_gas_station', 'p08_partition_labels']:
    print(f"  {name}")

## 1. Baseline — `p01_max_profit_stock`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert max_profit_stock([7, 1, 5, 3, 6, 4]) == 5
assert max_profit_stock([7, 6, 4, 3, 1]) == 0
assert max_profit_stock([]) == 0
assert max_profit_stock([5]) == 0
# Buying and selling the same day is not a trade.
assert max_profit_stock([3, 3]) == 0
assert max_profit_stock([1, 2]) == 1
# The minimum must come strictly before the maximum.
assert max_profit_stock([9, 1]) == 0
assert max_profit_stock([2, 4, 1]) == 2
# Cross-check against brute force.
for data in ([7, 1, 5, 3, 6, 4], [2, 4, 1], [1, 2, 3, 4], [4, 3, 2, 1], [3, 2, 6, 5, 0, 3]):
    brute = max(
        (data[j] - data[i] for i in range(len(data)) for j in range(i + 1, len(data))),
        default=0,
    )
    assert max_profit_stock(data) == max(brute, 0), data
# O(n).
assert max_profit_stock(list(range(100_000))) == 99_999

print("all assertions held")

## 2. Predict before you run

Four meetings: (1,10), (2,3), (4,5), (6,7). Predict how many non-overlapping meetings fit. Then predict what a greedy that sorts by START time selects first, and how many it ends up with.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]
# Touching intervals merge.
assert merge_intervals([(1, 4), (4, 5)]) == [(1, 5)]
assert merge_intervals([]) == []
assert merge_intervals([(1, 5)]) == [(1, 5)]
# A fully nested interval must not shrink the result.
assert merge_intervals([(1, 10), (2, 3)]) == [(1, 10)]
assert merge_intervals([(1, 4), (2, 3)]) == [(1, 4)]
# Input order must not matter.
assert merge_intervals([(15, 18), (1, 3), (8, 10), (2, 6)]) == [(1, 6), (8, 10), (15, 18)]
# No overlaps at all.
assert merge_intervals([(1, 2), (3, 4)]) == [(1, 2), (3, 4)]
# Everything collapses to one.
assert merge_intervals([(1, 100), (2, 3), (4, 5), (50, 60)]) == [(1, 100)]
# Identical intervals.
assert merge_intervals([(1, 2), (1, 2)]) == [(1, 2)]
# The output must be disjoint and sorted.
data = [(5, 7), (1, 4), (3, 3), (9, 12), (11, 20), (0, 0)]
got = merge_intervals(data)
assert got == sorted(got)
assert all(a[1] < b[0] for a, b in zip(got, got[1:]))

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert max_non_overlapping([(1, 3), (2, 4), (3, 5)]) == 2
assert max_non_overlapping([]) == 0
assert max_non_overlapping([(1, 2)]) == 1
# Touching intervals are both selectable.
assert max_non_overlapping([(1, 2), (2, 3), (3, 4)]) == 3
# All identical: only one can be taken.
assert max_non_overlapping([(1, 5), (1, 5), (1, 5)]) == 1
# THE case sorting by start gets wrong: start-order takes (1,10), # first and then only 1 fits, giving 1 instead of 3.
assert max_non_overlapping([(1, 10), (2, 3), (4, 5), (6, 7)]) == 3
# Nested intervals: take the innermost.
assert max_non_overlapping([(1, 100), (2, 3)]) == 1
# Disjoint intervals: take them all.
assert max_non_overlapping([(1, 2), (3, 4), (5, 6)]) == 3
# Cross-check against exhaustive subset search.
import itertools
for data in (
    [(1, 3), (2, 4), (3, 5)],
    [(1, 10), (2, 3), (4, 5), (6, 7)],
    [(0, 2), (1, 4), (3, 5), (4, 6)],
    [(1, 2), (2, 3)],
):
    best = 0
    for r in range(len(data) + 1):
        for combo in itertools.combinations(sorted(data, key=lambda iv: iv[1]), r):
            if all(a[1] <= b[0] for a, b in zip(combo, combo[1:])):
                best = max(best, r)
    assert max_non_overlapping(data) == best, data

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. Sort by start to merge. Sort by end to schedule. Getting it backwards is silent.
2. If you cannot prove the greedy choice is safe, use DP - slower and always correct.
3. A greedy scan's precondition is a check you owe the caller.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem